In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3

In [2]:
url = "http://quotes.toscrape.com/"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

records = []
for item in soup.find_all('div', class_='quote'):
    quote = item.find('span', class_='text').get_text().strip("“”")
    author = item.find('small', class_='author').get_text()
    tags = ", ".join([t.get_text() for t in item.find_all('a', class_='tag')])
    
    records.append({
        'Quote': quote,
        'Author': author,
        'Tags': tags
    })

df = pd.DataFrame(records)

In [3]:
db_name = 'web_mining.db'
conn = sqlite3.connect(db_name)

In [4]:
df.to_sql('web_quotes', conn, if_exists='replace', index=False)

10

In [5]:
df.to_csv('extracted_quotes.csv', index=False)

In [6]:
print("--- Querying Extracted Data from SQLite Database ---")
db_data = pd.read_sql_query("SELECT Author, Quote, Tags FROM web_quotes LIMIT 5", conn)
print(db_data)

conn.close()

--- Querying Extracted Data from SQLite Database ---
            Author                                              Quote  \
0  Albert Einstein  The world as we have created it is a process o...   
1     J.K. Rowling  It is our choices, Harry, that show what we tr...   
2  Albert Einstein  There are only two ways to live your life. One...   
3      Jane Austen  The person, be it gentleman or lady, who has n...   
4   Marilyn Monroe  Imperfection is beauty, madness is genius and ...   

                                           Tags  
0        change, deep-thoughts, thinking, world  
1                            abilities, choices  
2  inspirational, life, live, miracle, miracles  
3              aliteracy, books, classic, humor  
4                    be-yourself, inspirational  
